# Telco Customer Churn — Analysis Notebook

This notebook covers the exploratory-analysis side of the project. The production training logic lives in `src/train.py`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
DATA_PATH


## 1. Load the data

In [ ]:
if not DATA_PATH.exists():
    print('Dataset not found. Run: python src/download_data.py')
else:
    df = pd.read_csv(DATA_PATH)
    display(df.head())
    print('Shape:', df.shape)


## 2. Inspect the dataset

In [ ]:
df.info()
display(df.describe(include='all').T)


## 3. Clean `TotalCharges` and inspect missing values

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
display(df.isna().sum().sort_values(ascending=False).head(10))
print('Duplicate rows:', df.duplicated().sum())


## 4. Target distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
display(churn_counts)
print('Churn rate:', (df['Churn'].eq('Yes').mean()))

ax = churn_counts.plot(kind='bar', title='Customer Churn Distribution')
ax.set_xlabel('Churn')
ax.set_ylabel('Customers')
plt.show()


## 5. Churn rate by contract

In [ ]:
contract_churn = (
    df.assign(ChurnFlag=df['Churn'].eq('Yes').astype(int))
      .groupby('Contract')['ChurnFlag']
      .mean()
      .sort_values(ascending=False)
)
display(contract_churn)
contract_churn.plot(kind='bar', title='Churn Rate by Contract')
plt.ylabel('Churn rate')
plt.show()


## 6. Churn rate by internet service

In [ ]:
internet_churn = (
    df.assign(ChurnFlag=df['Churn'].eq('Yes').astype(int))
      .groupby('InternetService')['ChurnFlag']
      .mean()
      .sort_values(ascending=False)
)
display(internet_churn)
internet_churn.plot(kind='bar', title='Churn Rate by Internet Service')
plt.ylabel('Churn rate')
plt.show()


## 7. Tenure and churn

In [ ]:
for label in ['No', 'Yes']:
    df.loc[df['Churn'].eq(label), 'tenure'].plot(
        kind='hist', bins=20, alpha=0.5, label=label
    )
plt.title('Tenure Distribution by Churn')
plt.xlabel('Tenure (months)')
plt.legend(title='Churn')
plt.show()


## 8. Monthly charges and churn

In [ ]:
display(df.groupby('Churn')['MonthlyCharges'].describe())
df.boxplot(column='MonthlyCharges', by='Churn')
plt.suptitle('')
plt.title('Monthly Charges by Churn')
plt.ylabel('Monthly Charges')
plt.show()


## 9. Train the production model

The reusable training code is intentionally kept outside the notebook. This is more professional than making deployment depend on notebook cells.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(PROJECT_ROOT / 'src' / 'train.py')], check=True)


## 10. Next step

Run the application from a terminal:

```bash
streamlit run app.py
```